# Open-set refusal on EVA02 OOF (inner CV)

Closed-test candidate mode on identity-disjoint 5-fold OOF of LLM2CLIP EVA02-L-14-336 (`runs/cv/eva02_trial23`). Raw embeddings, no TTA, no postproc. `camera_id` / `vehicle_id` are labels for building pairs only, never features.

**Protocol (nested 5-fold, not a 20% identity hold-out).** For test fold `f`:

1. Build 50/50 pairs on every fold: same query vs full gallery (`y=1`) and vs that identity stripped from the gallery (`y=0`). Record whether cosine top-1 is the query identity (`top`).
2. Inner CV on the other four folds: train CatBoost/TabM on three, score the fourth. Cosine needs no fit.
3. Freeze each head's contest operating point (`0.7 × F1 + 0.3 × TNR`), rank-average weights, and vote rule on the pooled inner scores. Contest F1 is top-1 identity; extra ranks are ignored.
4. Retrain CatBoost/TabM on all four folds and score fold `f` with that fold's inner-CV thresholds. Do not reuse a threshold that saw fold `f`.
5. Outer-OOF metrics concatenate those per-fold accept/refuse decisions and compute one micro TP/FP/FN/TN. The serving threshold is the mean of the five inner-CV thresholds, estimated after this procedure.

Prevalence here is balanced 50/50, not the hidden-test 20% open-set mix. TNR is computed only on the stripped-gallery (no-match) queries. F1 and TNR are **micro** (one confusion matrix over all queries, not a per-query or per-identity mean). F1 is query-level: TP requires a gallery match **and** a correct top-1; a closed query with the wrong top-1 is FP. Candidates below rank-1 do not affect F1 or TNR.

| Head | Score |
| --- | --- |
| Cosine | max query–gallery cosine |
| CatBoost | `P(match exists)` |
| TabM | mean sigmoid over `k` BatchEnsemble members |
| rank-mean | equal-weight rank average of the three |
| rank-weighted | inner-CV nonnegative weights on those ranks |
| majority | accept if at least two heads accept |
| unanimous | accept only if all three accept |

Contest metrics: **`0.7 × F1 + 0.3 × TNR` at the frozen inner-CV threshold**, plus F1, TNR, and PR-AUC.


In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "train.csv").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [1]:
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve

from postproc.retrieval import normalize
from refusal import (
    STAT_NAMES,
    balanced_pack,
    candidate_metrics,
    decide,
    decision_metrics,
    feature_names,
    fit_boosting,
    fit_rank_weights,
    fit_tabm,
    predict_boosting,
    predict_tabm,
    rank_average,
    ranking_metrics,
    select_threshold,
    sweep_thresholds,
    vote_fraction,
)

plt.rcParams.update(
    {
        "figure.figsize": (10, 4),
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)


def display(obj):
    print(obj if isinstance(obj, str) else obj.to_string() if hasattr(obj, "to_string") else obj)


CV = ROOT / "runs" / "cv" / "eva02_trial23"
FIG_DIR = ROOT / "notebooks" / "eva02" / "refusal_analysis_figs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
K_RETR = 10
HEADS = ("cosine", "catboost", "tabm")
print("ROOT", ROOT)
print("CV", CV)

ROOT /data/projects/ryzhichkin/ReID
CV /data/projects/ryzhichkin/ReID/runs/cv/eva02_trial23


## Load OOF folds and 50/50 packs


In [2]:
def load_pack(directory):
    oof = pd.read_csv(directory / "oof.csv", dtype={"image_id": str})
    query = pd.read_csv(directory / "query.csv", dtype={"image_id": str})
    gallery = pd.read_csv(directory / "gallery.csv", dtype={"image_id": str})
    emb = normalize(np.load(directory / "embeddings.npy"))
    lookup = {image_id: i for i, image_id in enumerate(oof.image_id)}
    qe = emb[[lookup[i] for i in query.image_id]]
    ge = emb[[lookup[i] for i in gallery.image_id]]
    return {"query": query, "gallery": gallery, "qe": qe, "ge": ge}


FOLDS = {fold: load_pack(CV / f"fold{fold}" / "val") for fold in range(5)}
PACKS = {}
for fold, pack in FOLDS.items():
    X, y, cos, top = balanced_pack(
        pack["qe"],
        pack["ge"],
        pack["query"].vehicle_id.to_numpy(),
        pack["gallery"].vehicle_id.to_numpy(),
        k=K_RETR,
        with_embeddings=True,
    )
    PACKS[fold] = {"X": X, "y": y, "cos": cos, "top": top}
    print(
        fold,
        "pairs",
        len(y),
        "positives",
        float(y.mean()),
        "top1",
        float(top[y.astype(bool)].mean()),
        "dim",
        X.shape[1],
    )

0 pairs 618 positives 0.5 top1 0.8381877022653722 dim 535
1 pairs 616 positives 0.5 top1 0.8409090909090909 dim 535
2 pairs 616 positives 0.5 top1 0.8376623376623377 dim 535
3 pairs 616 positives 0.5 top1 0.827922077922078 dim 535
4 pairs 616 positives 0.5 top1 0.8668831168831169 dim 535


## Nested 5-fold fit

Inner models see only three of the four training folds. Thresholds and blend weights never touch the outer test fold. Each outer fold keeps its own inner-CV accept/refuse mask.


In [3]:
def concat_folds(folds, key):
    return np.concatenate([PACKS[fold][key] for fold in folds], 0)


def fit_models(train_folds, seed, epochs):
    X, y = concat_folds(train_folds, "X"), concat_folds(train_folds, "y")
    cat = fit_boosting(X, y, iterations=200, depth=4, learning_rate=0.08, seed=seed)
    tabm = fit_tabm(
        X,
        y,
        epochs=epochs,
        batch_size=128,
        patience=6,
        hidden=(64, 32),
        dropout=0.1,
        k=8,
        seed=seed,
        val_frac=0.15,
    )
    return cat, tabm


def predict_fold(cat, tabm, fold):
    return {
        "y": PACKS[fold]["y"],
        "top": PACKS[fold]["top"],
        "cosine": PACKS[fold]["cos"],
        "catboost": predict_boosting(cat, PACKS[fold]["X"]),
        "tabm": predict_tabm(tabm, PACKS[fold]["X"]),
    }


def summarize(name, y, scores, threshold, top):
    return {"method": name, **candidate_metrics(y, scores, threshold, top), **ranking_metrics(y, scores)}


ROWS = []
OOF: dict[str, dict[str, Any]] = {
    name: {"y": [], "top": [], "scores": [], "threshold": [], "accept": []}
    for name in (
        "always_accept",
        "cosine",
        "catboost",
        "tabm",
        "rank_mean",
        "rank_weighted",
        "majority",
        "unanimous",
    )
}
BLEND_W = []
MODELS: dict[int, dict[str, Any]] = {}
for test_fold in range(5):
    others = [fold for fold in range(5) if fold != test_fold]
    inner_y = []
    inner_top = []
    inner = {name: [] for name in HEADS}
    for val_fold in others:
        train_inner = [fold for fold in others if fold != val_fold]
        cat_i, tabm_i = fit_models(train_inner, seed=10 * test_fold + val_fold, epochs=12)
        pred_i = predict_fold(cat_i, tabm_i, val_fold)
        inner_y.append(pred_i["y"])
        inner_top.append(pred_i["top"])
        for name in HEADS:
            inner[name].append(pred_i[name])
    y_in = np.concatenate(inner_y)
    top_in = np.concatenate(inner_top)
    s_in = {name: np.concatenate(inner[name]) for name in HEADS}
    t_head = {name: select_threshold(y_in, s_in[name], top_in, kind="contest") for name in HEADS}
    channels_in = [s_in[name] for name in HEADS]
    t_rank = select_threshold(y_in, rank_average(channels_in), top_in, kind="contest")
    weights, t_weighted = fit_rank_weights(y_in, channels_in, top_in, grid=4)
    head_t = [t_head[name]["threshold"] for name in HEADS]
    votes_in = vote_fraction(channels_in, head_t)
    t_vote = select_threshold(y_in, votes_in, top_in, kind="contest")

    cat, tabm = fit_models(others, seed=test_fold, epochs=16)
    ev = predict_fold(cat, tabm, test_fold)
    channels = [ev[name] for name in HEADS]
    rank = rank_average(channels)
    rank_w = rank_average(channels, weights=weights)
    votes = vote_fraction(channels, head_t)
    methods = {
        "always_accept": (ev["cosine"], -1.0),
        "cosine": (ev["cosine"], t_head["cosine"]["threshold"]),
        "catboost": (ev["catboost"], t_head["catboost"]["threshold"]),
        "tabm": (ev["tabm"], t_head["tabm"]["threshold"]),
        "rank_mean": (rank, t_rank["threshold"]),
        "rank_weighted": (rank_w, t_weighted["threshold"]),
        "majority": (votes, 2.0 / 3.0),
        "unanimous": (votes, 1.0),
    }
    print(
        f"fold {test_fold}: test pairs {len(ev['y'])} inner {len(y_in)} "
        f"weights {np.round(weights, 3)} vote-t {t_vote['threshold']:.2f}"
    )
    MODELS[test_fold] = {"catboost": cat, "tabm": tabm, "eval": ev, "t_head": t_head, "weights": weights}
    BLEND_W.append(weights)
    for name, (pred, thr) in methods.items():
        row = summarize(name, ev["y"], pred, thr, ev["top"])
        row["fold"] = test_fold
        ROWS.append(row)
        OOF[name]["y"].append(ev["y"])
        OOF[name]["top"].append(ev["top"])
        OOF[name]["scores"].append(pred)
        OOF[name]["threshold"].append(thr)
        OOF[name]["accept"].append(decide(pred, thr))

for name in OOF:
    OOF[name]["y"] = np.concatenate(OOF[name]["y"])
    OOF[name]["top"] = np.concatenate(OOF[name]["top"])
    OOF[name]["scores"] = np.concatenate(OOF[name]["scores"])
    OOF[name]["accept"] = np.concatenate(OOF[name]["accept"])

table = pd.DataFrame(ROWS)
cols = [
    "fold",
    "method",
    "threshold",
    "contest",
    "f1",
    "precision",
    "recall",
    "tnr",
    "fp_closed",
    "pr_auc",
    "n_open",
    "n_closed",
]
display(table[cols])

fold 0: test pairs 618 inner 2464 weights [1. 0. 0.] vote-t 0.67
fold 1: test pairs 616 inner 2466 weights [1. 0. 0.] vote-t 0.67
fold 2: test pairs 616 inner 2466 weights [0.75 0.   0.25] vote-t 0.67
fold 3: test pairs 616 inner 2466 weights [1. 0. 0.] vote-t 0.67
fold 4: test pairs 616 inner 2466 weights [0.6 0.4 0. ] vote-t 0.67
    fold         method  threshold  contest     f1  precision  recall    tnr  fp_closed  pr_auc  n_open  n_closed
0      0  always_accept    -1.0000   0.4135 0.5906     0.4191  1.0000 0.0000         50  0.8708     309       309
1      0         cosine     0.7246   0.7567 0.6913     0.8018  0.6075 0.9094         16  0.8708     309       309
2      0       catboost     0.7186   0.7618 0.7000     0.7982  0.6233 0.9061         17  0.8739     309       309
3      0           tabm     0.5757   0.7628 0.7179     0.7510  0.6877 0.8673         24  0.8699     309       309
4      0      rank_mean     0.5765   0.7679 0.7239     0.7586  0.6923 0.8706         23  0.8747 

## Aggregate contest metrics

Mean ± std over the five outer folds (each fold uses only its inner-CV threshold). Outer-OOF micro metrics concatenate those decisions; they do not re-apply a pooled threshold. Serving thresholds are the mean inner-CV points, reported separately. Always-accept TNR is 0. F1 penalizes a closed query whose top-1 identity is wrong.


In [4]:
order = ["always_accept", "cosine", "catboost", "tabm", "rank_mean", "rank_weighted", "majority", "unanimous"]
agg = (
    table.groupby("method")[["contest", "f1", "precision", "recall", "tnr", "pr_auc", "threshold"]]
    .agg(["mean", "std"])
    .loc[order]
)
display(agg)

pooled_rows = []
deploy_rows = []
for name in order:
    y = OOF[name]["y"]
    top = OOF[name]["top"]
    scores = OOF[name]["scores"]
    row = {
        "method": name + "/oof",
        **decision_metrics(y, OOF[name]["accept"], top),
        **ranking_metrics(y, scores),
    }
    pooled_rows.append(row)
    deploy_t = float(np.mean(OOF[name]["threshold"]))
    deploy = summarize(name + "/deploy_t", y, scores, deploy_t, top)
    deploy_rows.append(deploy)
pooled = pd.DataFrame(pooled_rows)
deploy = pd.DataFrame(deploy_rows)
display(
    pooled[
        ["method", "contest", "f1", "precision", "recall", "tnr", "fp_closed", "pr_auc", "n_open", "n_closed"]
    ]
)
display(deploy[["method", "threshold", "contest", "f1", "tnr"]])
print("blend weights mean", np.round(np.mean(BLEND_W, 0), 3), "std", np.round(np.std(BLEND_W, 0), 3))
print(
    "serving cosine",
    round(float(np.mean(OOF["cosine"]["threshold"])), 4),
    "catboost",
    round(float(np.mean(OOF["catboost"]["threshold"])), 4),
)

              contest            f1        precision        recall           tnr        pr_auc        threshold       
                 mean    std   mean    std      mean    std   mean    std   mean    std   mean    std      mean    std
method                                                                                                                
always_accept  0.4149 0.0050 0.5927 0.0072    0.4212 0.0073 1.0000 0.0000 0.0000 0.0000 0.8789 0.0137   -1.0000 0.0000
cosine         0.7635 0.0116 0.7248 0.0318    0.7617 0.0473 0.7028 0.1067 0.8540 0.0694 0.8789 0.0137    0.7011 0.0136
catboost       0.7596 0.0137 0.7195 0.0318    0.7595 0.0500 0.6953 0.1073 0.8533 0.0706 0.8750 0.0079    0.6719 0.0412
tabm           0.7418 0.0258 0.7063 0.0270    0.7332 0.0670 0.6998 0.1249 0.8248 0.1172 0.8677 0.0083    0.5926 0.0468
rank_mean      0.7716 0.0105 0.7332 0.0136    0.7554 0.0083 0.7126 0.0232 0.8611 0.0072 0.8787 0.0105    0.5654 0.0074
rank_weighted  0.7714 0.0124 0.7352 0.0120    0.

## Inner-CV cosine threshold (fold 0)

The sweep below is the inner OOF of fold 0's training folds. The vertical line is the frozen contest point (`0.7 × F1 + 0.3 × TNR`) applied to fold 0's 50/50 pairs. F1 uses contest top-1 identity, not pair-level hits. max-F1 / Youden / F1×TNR are shown only as alternate rules.


In [5]:
fold0_others = [1, 2, 3, 4]
inner_y = []
inner_cos = []
inner_top = []
for val_fold in fold0_others:
    inner_y.append(PACKS[val_fold]["y"])
    inner_cos.append(PACKS[val_fold]["cos"])
    inner_top.append(PACKS[val_fold]["top"])
y_in = np.concatenate(inner_y)
cos_in = np.concatenate(inner_cos)
top_in = np.concatenate(inner_top)
y0 = PACKS[0]["y"]
cos0 = PACKS[0]["cos"]
top0 = PACKS[0]["top"]
rules = {
    "contest": select_threshold(y_in, cos_in, top_in, kind="contest"),
    "max_f1": select_threshold(y_in, cos_in, top_in, kind="max_f1"),
    "youden": select_threshold(y_in, cos_in, top_in, kind="youden"),
    "f1_tnr": select_threshold(y_in, cos_in, top_in, kind="f1_tnr"),
}
rule_tab = []
for name, picked in rules.items():
    row = candidate_metrics(y0, cos0, picked["threshold"], top0)
    row["rule"] = name
    row["cal_contest"] = picked["contest"]
    row["cal_f1"] = picked["f1"]
    row["cal_tnr"] = picked["tnr"]
    rule_tab.append(row)
display(
    pd.DataFrame(rule_tab)[
        [
            "rule",
            "threshold",
            "cal_contest",
            "cal_f1",
            "cal_tnr",
            "contest",
            "f1",
            "tnr",
            "precision",
            "recall",
            "fp_closed",
        ]
    ]
)

sweep_cal = pd.DataFrame(sweep_thresholds(y_in, cos_in, top_in))
sweep_ev = pd.DataFrame(sweep_thresholds(y0, cos0, top0))
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
ax[0].plot(sweep_cal.threshold, sweep_cal.contest, label="0.7 F1 + 0.3 TNR")
ax[0].plot(sweep_cal.threshold, sweep_cal.f1, label="F1")
ax[0].plot(sweep_cal.threshold, sweep_cal.tnr, label="TNR")
ax[0].axvline(rules["contest"]["threshold"], color="C3", ls="--", label="contest")
ax[0].set_title("inner CV (folds 1–4)")
ax[0].set_xlabel("cosine threshold")
ax[0].legend()
ax[1].plot(sweep_ev.threshold, sweep_ev.contest, label="0.7 F1 + 0.3 TNR")
ax[1].plot(sweep_ev.threshold, sweep_ev.f1, label="F1")
ax[1].plot(sweep_ev.threshold, sweep_ev.tnr, label="TNR")
ax[1].axvline(rules["contest"]["threshold"], color="C3", ls="--", label="frozen")
ax[1].set_title("outer fold 0, 50/50 pairs")
ax[1].set_xlabel("cosine threshold")
ax[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "threshold_sweep.png", dpi=140, bbox_inches="tight")
plt.show()

      rule  threshold  cal_contest  cal_f1  cal_tnr  contest     f1    tnr  precision  recall  fp_closed
0  contest     0.7246       0.7701  0.7220   0.8823   0.7567 0.6913 0.9094     0.8018  0.6075         16
1   max_f1     0.6612       0.7593  0.7508   0.7792   0.7721 0.7521 0.8188     0.7237  0.7829         28
2   youden     0.6612       0.7593  0.7508   0.7792   0.7721 0.7521 0.8188     0.7237  0.7829         28
3   f1_tnr     0.7246       0.7701  0.7220   0.8823   0.7567 0.6913 0.9094     0.8018  0.6075         16


Saved figure:

![threshold sweep](refusal_analysis_figs/threshold_sweep.png)


## Score separation on pooled OOF pairs


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, name, title in zip(axes, HEADS, ("max cosine", "CatBoost", "TabM"), strict=True):
    y = OOF[name]["y"].astype(bool)
    s = OOF[name]["scores"]
    ax.hist(s[y], bins=30, alpha=0.7, label="has match", density=True)
    ax.hist(s[~y], bins=30, alpha=0.7, label="no match", density=True)
    ax.axvline(np.mean(OOF[name]["threshold"]), color="k", ls="--", label="deploy t")
    ax.set_title(title)
    ax.set_xlabel("score")
axes[0].set_ylabel("density")
axes[0].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "score_hist.png", dpi=140, bbox_inches="tight")
plt.show()

Saved figure:

![score histograms](refusal_analysis_figs/score_hist.png)


## Precision–recall, including ensembles


In [7]:
fig, ax = plt.subplots(figsize=(6.5, 5.2))
for name, title in (
    ("cosine", "cosine"),
    ("catboost", "CatBoost"),
    ("tabm", "TabM"),
    ("rank_mean", "rank-mean"),
    ("rank_weighted", "rank-weighted"),
    ("majority", "majority votes"),
):
    y = OOF[name]["y"]
    s = OOF[name]["scores"]
    p, r, _ = precision_recall_curve(y, s)
    ap = ranking_metrics(y, s)["pr_auc"]
    ax.plot(r, p, label=f"{title}  PR-AUC={ap:.3f}")
ax.set_xlabel("recall (has-match)")
ax.set_ylabel("precision")
ax.set_title("Concatenated outer-OOF PR curves")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "pr_curves.png", dpi=140, bbox_inches="tight")
plt.show()

Saved figure:

![PR curves](refusal_analysis_figs/pr_curves.png)


## Head-to-head, with ensembles last


In [8]:
fig, ax = plt.subplots(figsize=(10, 4.2))
methods = order
labels = [
    "always accept",
    "cosine t",
    "CatBoost",
    "TabM",
    "rank-mean",
    "rank-weighted",
    "majority",
    "unanimous",
]
x = np.arange(len(methods))
width = 0.2
means = table.groupby("method")[["contest", "f1", "tnr", "pr_auc"]].mean().loc[methods]
for i, metric in enumerate(("contest", "f1", "tnr", "pr_auc")):
    label = {"contest": "0.7 F1+0.3 TNR", "f1": "F1", "tnr": "TNR", "pr_auc": "PR-AUC"}[metric]
    ax.bar(x + (i - 1.5) * width, means[metric], width, label=label)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("5-fold mean")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "metrics_bars.png", dpi=140, bbox_inches="tight")
plt.show()

Saved figure:

![metrics bars](refusal_analysis_figs/metrics_bars.png)


## CatBoost statistics


In [9]:
names = feature_names(FOLDS[0]["qe"].shape[1])
imps = np.stack([MODELS[fold]["catboost"].get_feature_importance() for fold in range(5)]).mean(0)
imp = pd.DataFrame({"feature": names, "importance": imps}).sort_values("importance", ascending=False)
display(imp.head(15))
stat_imp = imp[imp.feature.isin(STAT_NAMES)].copy()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(stat_imp.feature[::-1], stat_imp.importance[::-1])
ax.set_title("CatBoost mean importance, retrieval statistics")
fig.tight_layout()
fig.savefig(FIG_DIR / "catboost_stats.png", dpi=140, bbox_inches="tight")
plt.show()

          feature  importance
0            top1     17.8751
9       mean_top3      4.4955
1            top2      4.0139
13   pairwise_std      3.2504
6           std_k      3.1965
11        entropy      3.0831
22    margin_rest      2.7342
10      mean_top5      2.6542
16        n_ge_07      2.5680
14  query_to_mean      1.1919
5          mean_k      1.1657
2            top3      1.0022
12  pairwise_mean      0.8578
7        median_k      0.7570
47           q_24      0.6341


Saved figure:

![CatBoost stats](refusal_analysis_figs/catboost_stats.png)


## Takeaway

- Local refusal metrics are nested 5-fold OOF on 50/50 stripped-identity pairs, not a 20% gallery hold-out.
- The operating point maximizes the contest score `0.7 × F1 + 0.3 × TNR` on inner-CV scores. max-F1 alone can pick a point with TNR 0.
- Contest F1 is query-level: TP = has match and top-1 identity is correct. Extra candidates do not count as extra FP. Closed wrong top-1 is FP (`fp_closed`).
- TNR uses only no-match queries. Always-accept still has TNR 0.
- Outer-OOF accept/refuse uses each fold's own inner-CV threshold. Serving cosine/CatBoost thresholds are the mean of those five inner points, not a threshold fit on the evaluated queries.
- Rank-average / majority combine cosine, CatBoost, and TabM after each head is calibrated on inner CV. Serving `eva02_ensemble` is the streaming-safe unanimous vote of the frozen cosine and CatBoost heads.
- Hidden-test prevalence is ~20% open. This notebook's pairs are 50/50, so a still-higher TNR may help if the leaderboard mix is more open than the OOF pairs.
